# Chương 2 - Xây dựng dữ liệu bảng và phương pháp giả bảng từ dữ liệu chéo lặp lại

*Ứng dụng trong các bộ dữ liệu cấp hộ gia đình tại Việt Nam (VARHS và VHLSS).*

Notebook này tổng hợp các đoạn mã STATA của mục **3.1 (VARHS)** và mục **3.2 (VHLSS)** trong chương, kèm phần thuyết minh ngắn cho từng bước. Mỗi ô lệnh phía dưới tương ứng với một đoạn mã trong sách.

---

## Độc giả cần lưu ý chỉnh lại đường dẫn trước khi chạy trên máy tính cá nhân

Các ô `cd "..."` bên dưới đang trỏ tới đường dẫn cục bộ của tác giả (`D:\Một số phương pháp...`). Trong trường hợp độc giả tải file về máy tính cá nhân để chạy, hãy đổi mỗi đường dẫn `cd` thành đường dẫn tới thư mục `data/` đã tải xuống.

Ví dụ, nếu thư mục dữ liệu được lưu vào `C:\Users\vn-data\panel-data-book`, cần đổi:

`cd "D:\Một số phương pháp hiện đại xử lý và phân tích dữ liệu kinh tế\VARHS\2008-16Clean\2014_new"`

thành:

`cd "C:\Users\vn-data\panel-data-book\data\VARHS\2008-16Clean\2014_new"`

Độc giả có thể dùng tính năng **Find & Replace** (Ctrl+H) trong Stata Do-file Editor / VS Code / Jupyter để đổi một lượt tất cả các đường dẫn.

Ngoài ra, độc giả cần cài đặt **stata_kernel** để có thể chạy các câu lệnh STATA bằng Jupyter Notebook.

# 3.1. Quy trình tạo dữ liệu bảng từ dữ liệu VARHS

Quy trình gồm bốn bước: (i) xác định mã định danh cho cấp độ phân tích thấp nhất, (ii) lọc và tính toán các trường thông tin cần thiết, (iii) hợp nhất dữ liệu giữa các năm và đồng nhất mã định danh, (iv) khai báo cấu trúc bảng.

## 3.1.1. Tạo mã định danh

Mã định danh hộ trong VARHS được xây dựng bằng cách kết hợp các mã `tinh`, `quan`, `xa` và `ma_h0`. Có thể sử dụng 2 cách sau:

- **Dạng số:** nối các mã thành một số duy nhất, cần biết số chữ số tối đa của mỗi thành phần để các nhóm chữ số không bị chồng lấp.
- **Dạng chuỗi:** nối các mã bằng dấu phân tách (ví dụ `_`); tránh được vấn đề số chữ số khác nhau giữa các năm, nên cách này được dùng xuyên suốt cho phần xây dựng bảng phía dưới.

In [115]:
* Thiết lập thư mục làm việc
cd "D:\Một số phương pháp hiện đại xử lý và phân tích dữ liệu kinh tế\VARHS\2008-16Clean\2014_new"

* Tải lên dữ liệu bảng hỏi 2
use "Q2_New_14", clear


D:\Một số phương pháp hiện đại xử lý và phân tích dữ liệu kinh tế\VARHS\2008-16C
> lean\2014_new



In [116]:
* Kiểm tra các biến định danh
summarize tinh_2014
summarize quan_2014
summarize xa_2014
summarize ma_h0_2014



    Variable |        Obs        Mean    Std. dev.       Min        Max
-------------+---------------------------------------------------------
   tinh_2014 |     11,757    345.9511    212.7217        105        801


    Variable |        Obs        Mean    Std. dev.       Min        Max
-------------+---------------------------------------------------------
   quan_2014 |     11,757    14.76338     9.96309          1         50


    Variable |        Obs        Mean    Std. dev.       Min        Max
-------------+---------------------------------------------------------
     xa_2014 |     11,757     23.7382    15.85998          1         99


    Variable |        Obs        Mean    Std. dev.       Min        Max
-------------+---------------------------------------------------------
  ma_h0_2014 |     11,757    15.77069    21.50732          1         99


In [117]:
* Tạo biến định danh hộ (dạng số)
gen hh_id_num = ma_h0_2014 + xa_2014*10^2 + quan_2014*10^4 + tinh_2014*10^6

In [118]:
* Tạo mã định danh hộ (dạng chuỗi-string)
gen hh_id_str = string(tinh_2014) + "_" + string(quan_2014) + "_" + string(xa_2014) + "_" + string(ma_h0_2014)

Với bảng dữ liệu cấp cá nhân, cách tạo mã định danh cũng có quy tắc tương tự và chỉ cần thêm số thứ tự của thành viên trong hộ (`p1stt_`).

In [119]:
* Tải lên dữ liệu bảng hỏi 1
use "Q1_New_14", clear

* Tạo mã định danh cá nhân (dạng chuỗi-string)
gen ind_id_str = string(tinh_2014) + "_" + string(quan_2014) + "_" + string(xa_2014) + "_" + string(ma_h0_2014) + "_" + string(p1stt_)

## 3.1.2. Gộp tệp nhằm tính toán liên tệp

Thông tin từ các bộ câu hỏi khác nhau được lưu trên các tệp khác nhau cho cùng một năm. Khi cần gộp, lưu ý **độ chi tiết (granularity)** của từng bảng: ví dụ bảng hỏi 2 năm 2014 có mỗi dòng tương ứng với một *mảnh đất* — phải `collapse` về cấp hộ trước, rồi mới `merge 1:1` với bảng cấp hộ khác.

In [120]:
* Thiết lập thư mục làm việc
cd "D:\Một số phương pháp hiện đại xử lý và phân tích dữ liệu kinh tế\VARHS\2008-16Clean\2014_new"

* Tải lên dữ liệu bảng hỏi 2
use "Q2_New_14", clear

gen hh_id_str = string(tinh_2014) + "_" + string(quan_2014) + "_" + string(xa_2014) + "_" + string(ma_h0_2014)


D:\Một số phương pháp hiện đại xử lý và phân tích dữ liệu kinh tế\VARHS\2008-16C
> lean\2014_new




In [121]:
* Lấy dữ liệu về tổng diện tích đất được nhà nước cấp cho MỖI hộ
gen area_given = p5q3a_ if p5q4_ == 1

(6,990 missing values generated)


In [122]:
collapse (sum) plot_given = area_given, by(hh_id_str)

In [123]:
keep hh_id_str plot_given
save "Q2_cleaned", replace



file Q2_cleaned.dta saved


In [124]:
use "Q8_New_14", clear

gen hh_id_str = string(tinh_2014) + "_" + string(quan_2014) + "_" + string(xa_2014) + "_" + string(ma_h0_2014)

In [125]:
* Lấy dữ liệu về số nợ còn lại
collapse (sum) loan = p41q7_, by(hh_id_str)

In [126]:
keep hh_id_str loan
save "Q8_cleaned", replace



file Q8_cleaned.dta saved


In [127]:
merge 1:1 hh_id_str using "Q2_cleaned"

* Kiểm tra kết quả merge
tab _merge



    Result                      Number of obs
    -----------------------------------------
    Not matched                         1,782
        from master                         0  (_merge==1)
        from using                      1,782  (_merge==2)

    Matched                               943  (_merge==3)
    -----------------------------------------


   Matching result from |
                  merge |      Freq.     Percent        Cum.
------------------------+-----------------------------------
         Using only (2) |      1,782       65.39       65.39
            Matched (3) |        943       34.61      100.00
------------------------+-----------------------------------
                  Total |      2,725      100.00


In [128]:
replace loan = 0 if _merge == 2

save "merged_data", replace


(1,782 real changes made)

file merged_data.dta saved


## 3.1.3. Quy trình xây dựng dữ liệu bảng cấp hộ

Vì mã định danh của hộ có thể thay đổi qua từng năm (do thay đổi địa lý hoặc cách gán mã), điều cốt lõi là cần **một mã định danh duy nhất xuyên suốt các năm**. UNU-WIDER đã đính kèm tệp `panelid` cho VARHS 2008–2016 để kết nối mã định danh theo từng năm về cùng một hộ. Do đó quy trình còn lại chỉ cần:

1. Xây dựng các bảng dữ liệu năm bao gồm các biến cần thiết (lặp lại quy trình 3.1.2 cho 2012, 2014, 2016).
2. Xử lý `panelid` để có mã `panel_id` thống nhất qua các năm.
3. Gộp các bảng năm vào, khai báo `xtset`, lọc về bảng cân bằng nếu cần.

### Bước 1: Chuẩn bị các bảng năm 2012, 2014, 2016

In [129]:
* ----------------- VARHS 2012 -----------------
cd "D:\Một số phương pháp hiện đại xử lý và phân tích dữ liệu kinh tế\VARHS\2008-16Clean"

use "2012_new\Q2_New_12", clear
gen hh_id_12 = string(tinh_2012) + "_" + string(quan_2012) + "_" + string(xa_2012) + "_" + string(ma_h0_2012)


D:\Một số phương pháp hiện đại xử lý và phân tích dữ liệu kinh tế\VARHS\2008-16C
> lean




In [130]:
gen area_given = p5q3a_ if p5q4_ == 1

(6,270 missing values generated)


In [131]:
collapse (sum) plot_given = area_given, by(hh_id_12)

In [132]:
keep hh_id_12 plot_given
save "panel_1216\middle_steps\Q2_cleaned", replace



file panel_1216\middle_steps\Q2_cleaned.dta saved


In [133]:
use "2012_new\Q8_New_12", clear
gen hh_id_12 = string(tinh_2012) + "_" + string(quan_2012) + "_" + string(xa_2012) + "_" + string(ma_h0_2012)

In [134]:
collapse (sum) loan = p39q7_, by(hh_id_12)

In [135]:
keep hh_id_12 loan
save "panel_1216\middle_steps\Q8_cleaned", replace



file panel_1216\middle_steps\Q8_cleaned.dta saved


In [136]:
merge 1:1 hh_id_12 using "panel_1216\middle_steps\Q2_cleaned"
tab _merge



    Result                      Number of obs
    -----------------------------------------
    Not matched                         1,662
        from master                         0  (_merge==1)
        from using                      1,662  (_merge==2)

    Matched                             1,097  (_merge==3)
    -----------------------------------------


   Matching result from |
                  merge |      Freq.     Percent        Cum.
------------------------+-----------------------------------
         Using only (2) |      1,662       60.24       60.24
            Matched (3) |      1,097       39.76      100.00
------------------------+-----------------------------------
                  Total |      2,759      100.00


In [137]:
replace loan = 0 if _merge == 2

(1,662 real changes made)


In [138]:
gen year = 12

In [139]:
drop _merge
save "panel_1216\merged_data_12", replace



file panel_1216\merged_data_12.dta saved


In [140]:
* ----------------- VARHS 2014 -----------------
use "2014_new\Q2_New_14", clear
gen hh_id_14 = string(tinh_2014) + "_" + string(quan_2014) + "_" + string(xa_2014) + "_" + string(ma_h0_2014)

In [141]:
gen area_given = p5q3a_ if p5q4_ == 1

(6,990 missing values generated)


In [142]:
collapse (sum) plot_given = area_given, by(hh_id_14)

In [143]:
keep hh_id_14 plot_given
save "panel_1216\middle_steps\Q2_cleaned", replace



file panel_1216\middle_steps\Q2_cleaned.dta saved


In [144]:
use "2014_new\Q8_New_14", clear
gen hh_id_14 = string(tinh_2014) + "_" + string(quan_2014) + "_" + string(xa_2014) + "_" + string(ma_h0_2014)

In [145]:
collapse (sum) loan = p41q7_, by(hh_id_14)

In [146]:
keep hh_id_14 loan
save "panel_1216\middle_steps\Q8_cleaned", replace



file panel_1216\middle_steps\Q8_cleaned.dta saved


In [147]:
merge 1:1 hh_id_14 using "panel_1216\middle_steps\Q2_cleaned"
tab _merge



    Result                      Number of obs
    -----------------------------------------
    Not matched                         1,782
        from master                         0  (_merge==1)
        from using                      1,782  (_merge==2)

    Matched                               943  (_merge==3)
    -----------------------------------------


   Matching result from |
                  merge |      Freq.     Percent        Cum.
------------------------+-----------------------------------
         Using only (2) |      1,782       65.39       65.39
            Matched (3) |        943       34.61      100.00
------------------------+-----------------------------------
                  Total |      2,725      100.00


In [148]:
replace loan = 0 if _merge == 2
gen year = 14


(1,782 real changes made)



In [149]:
drop _merge
save "panel_1216\merged_data_14", replace



file panel_1216\merged_data_14.dta saved


In [150]:
* ----------------- VARHS 2016 -----------------
use "2016_new\Q2_New_16", clear
gen hh_id_16 = string(tinh_2016) + "_" + string(quan_2016) + "_" + string(xa_2016) + "_" + string(ma_h0_2016)

In [151]:
gen area_given = p5q3a_ if p5q4_ == 1

(5,773 missing values generated)


In [152]:
collapse (sum) plot_given = area_given, by(hh_id_16)

In [153]:
keep hh_id_16 plot_given
save "panel_1216\middle_steps\Q2_cleaned", replace



file panel_1216\middle_steps\Q2_cleaned.dta saved


In [154]:
use "2016_new\Q8_New_16", clear
gen hh_id_16 = string(tinh_2016) + "_" + string(quan_2016) + "_" + string(xa_2016) + "_" + string(ma_h0_2016)

In [155]:
collapse (sum) loan = p40q7_, by(hh_id_16)

In [156]:
keep hh_id_16 loan
save "panel_1216\middle_steps\Q8_cleaned", replace



file panel_1216\middle_steps\Q8_cleaned.dta saved


In [157]:
merge 1:1 hh_id_16 using "panel_1216\middle_steps\Q2_cleaned"
tab _merge



    Result                      Number of obs
    -----------------------------------------
    Not matched                         1,901
        from master                         0  (_merge==1)
        from using                      1,901  (_merge==2)

    Matched                               768  (_merge==3)
    -----------------------------------------


   Matching result from |
                  merge |      Freq.     Percent        Cum.
------------------------+-----------------------------------
         Using only (2) |      1,901       71.23       71.23
            Matched (3) |        768       28.77      100.00
------------------------+-----------------------------------
                  Total |      2,669      100.00


In [158]:
replace loan = 0 if _merge == 2

(1,901 real changes made)


In [159]:
gen year = 16

In [160]:
drop _merge
save "panel_1216\merged_data_16", replace



file panel_1216\merged_data_16.dta saved


### Bước 2: Xử lý `panelid` để có `panel_id` thống nhất

In [161]:
use "panelid", clear

gen hh_id_12 = string(tinh_2012) + "_" + string(quan_2012) + "_" + string(xa_2012) + "_" + string(ma_h0_2012)
gen hh_id_14 = string(tinh_2014) + "_" + string(quan_2014) + "_" + string(xa_2014) + "_" + string(ma_h0_2014)
gen hh_id_16 = string(tinh_2016) + "_" + string(quan_2016) + "_" + string(xa_2016) + "_" + string(ma_h0_2016)

In [162]:
* Loại các hộ không xuất hiện trong cả 3 năm và các năm ngoài 2012-2016
drop if missing(tinh_2012) & missing(tinh_2014) & missing(tinh_2016)
keep if year == 12 | year == 14 | year == 16


(115 observations deleted)

(4,408 observations deleted)


In [163]:
* Mã định danh duy nhất cho từng hộ
egen panel_id = group(hh_id_12 hh_id_14 hh_id_16)

In [164]:
tempfile master merged12 merged14 merged16 final
save `master'



file C:\Users\viett\AppData\Local\Temp\ST_2fc0_00000e.tmp saved as .dta format


### Bước 3: Ghép `panelid` với từng năm rồi gộp toàn bộ

In [165]:
* ----- Năm 2012 -----
use `master', clear
keep if year == 12
preserve
keep if !missing(tinh_2012)
merge 1:1 hh_id_12 year using "panel_1216\merged_data_12.dta"
tempfile nonmiss12
save `nonmiss12'
restore
keep if missing(tinh_2012)
append using `nonmiss12'
save `merged12'



(5,394 observations deleted)


(0 observations deleted)


    Result                      Number of obs
    -----------------------------------------
    Not matched                             1
        from master                         1  (_merge==1)
        from using                          0  (_merge==2)

    Matched                             2,759  (_merge==3)
    -----------------------------------------


file C:\Users\viett\AppData\Local\Temp\ST_2fc0_00000k.tmp saved as .dta format


(2,760 observations deleted)

(label province already defined)

file C:\Users\viett\AppData\Local\Temp\ST_2fc0_00000f.tmp saved as .dta format


In [166]:
* ----- Năm 2014 -----
use `master', clear
keep if year == 14
preserve
keep if !missing(tinh_2014)
merge 1:1 hh_id_14 year using "panel_1216\merged_data_14.dta"
tempfile nonmiss14
save `nonmiss14'
restore
keep if missing(tinh_2014)
append using `nonmiss14'
save `merged14'



(5,429 observations deleted)


(0 observations deleted)


    Result                      Number of obs
    -----------------------------------------
    Not matched                             0
    Matched                             2,725  (_merge==3)
    -----------------------------------------


file C:\Users\viett\AppData\Local\Temp\ST_2fc0_00000m.tmp saved as .dta format


(2,725 observations deleted)

(label province already defined)

file C:\Users\viett\AppData\Local\Temp\ST_2fc0_00000g.tmp saved as .dta format


In [167]:
* ----- Năm 2016 -----
use `master', clear
keep if year == 16
preserve
keep if !missing(tinh_2016)
merge 1:1 hh_id_16 year using "panel_1216\merged_data_16.dta"
tempfile nonmiss16
save `nonmiss16'
restore
keep if missing(tinh_2016)
append using `nonmiss16'
save `merged16'



(5,485 observations deleted)


(0 observations deleted)


    Result                      Number of obs
    -----------------------------------------
    Not matched                             0
    Matched                             2,669  (_merge==3)
    -----------------------------------------


file C:\Users\viett\AppData\Local\Temp\ST_2fc0_00000o.tmp saved as .dta format


(2,669 observations deleted)

(label province already defined)

file C:\Users\viett\AppData\Local\Temp\ST_2fc0_00000h.tmp saved as .dta format


### Bước 4: Khai báo dữ liệu bảng

`xtset` lần đầu cho ra **bảng không cân bằng** vì có hộ chỉ xuất hiện 1–2 năm. Để tạo bảng cân bằng, ta cần đếm số lần mỗi hộ xuất hiện và lọc để có **bảng cân bằng**.

In [168]:
use `merged12', clear
append using `merged14'
append using `merged16'
save `final', replace



(label _merge already defined)
(label province already defined)

(label _merge already defined)
(label province already defined)

(file C:\Users\viett\AppData\Local\Temp\ST_2fc0_00000i.tmp not found)
file C:\Users\viett\AppData\Local\Temp\ST_2fc0_00000i.tmp saved as .dta format


In [169]:
keep panel_id year plot_given loan

In [170]:
xtset panel_id year


Panel variable: panel_id (unbalanced)
 Time variable: year, 12 to 16, but with gaps
         Delta: 1 unit


In [171]:
bysort panel_id: gen obs_count = _N
tab obs_count




  obs_count |      Freq.     Percent        Cum.
------------+-----------------------------------
          1 |         44        0.54        0.54
          2 |        124        1.52        2.06
          3 |      7,986       97.94      100.00
------------+-----------------------------------
      Total |      8,154      100.00


In [172]:
keep if obs_count == 3

xtset panel_id year


(168 observations deleted)


Panel variable: panel_id (strongly balanced)
 Time variable: year, 12 to 16, but with gaps
         Delta: 1 unit


# 3.2. Xây dựng bộ dữ liệu giả bảng (pseudo-panel) từ dữ liệu VHLSS

Vì VHLSS chỉ giữ một phần mẫu lặp lại qua các kỳ, khả năng theo cùng một hộ qua nhiều năm là hạn chế. Phương pháp **giả bảng** thay vì theo dõi từng hộ riêng lẻ sẽ nhóm các hộ vào **cohort** (theo đặc điểm bất biến theo thời gian) rồi lấy trung bình theo cohort trong từng năm.

Quy trình: (i) tạo mã cohort, (ii) lọc và tính các biến cần thiết, (iii) hợp nhất giữa các năm, (iv) khai báo bảng. Trong chương này, chúng ta sẽ xem xét ba phương án phân nhóm:
1. Theo nhóm năm sinh của chủ hộ
2. Theo nhóm năm sinh × giới tính chủ hộ
3. Theo đơn vị hành chính cấp tỉnh

Bộ dữ liệu minh họa lấy từ Nguyen và cộng sự (2021) với các năm đã được gộp sẵn vào một tệp.

## Tải dữ liệu và lọc biến nền

In [173]:
* Thiết lập thư mục làm việc
cd "D:\Một số phương pháp hiện đại xử lý và phân tích dữ liệu kinh tế\VHLSS"

use "VHLSS data", clear

* Mã định danh hộ (dạng chuỗi)
gen hh_id = string(tinh) + "_" + string(huyen) + "_" + string(xa) + "_" + string(diaban) + "_" + string(hoso)


D:\Một số phương pháp hiện đại xử lý và phân tích dữ liệu kinh tế\VHLSS

(Household expenditures: 2006 VHLSS)



In [174]:
* Năm sinh của chủ hộ = năm khảo sát − tuổi chủ hộ
gen birth_year = year - agehead

* Giữ lại các biến: thubq = thu nhập bình quân đầu người, tobacexp = chi tiêu thuốc lá
keep hh_id birth_year sexhead tinh thubq tobacexp year

## Phương án 1: Cohort theo nhóm năm sinh

Nhóm 5 năm từ 1935–1980, cộng với hai nhóm biên `<1935` và `≥1980`.

In [175]:
gen birth_cohort = ""

(55,971 missing values generated)


In [176]:
replace birth_cohort = "<1935" if birth_year < 1935

variable birth_cohort was str1 now str5
(2,898 real changes made)


In [177]:
replace birth_cohort = string(floor(birth_year/5)*5) + "-" + string(floor(birth_year/5)*5 + 5) if birth_year >= 1935 & birth_year < 1980

variable birth_cohort was str5 now str9
(48,475 real changes made)


In [178]:
replace birth_cohort = ">=1980" if birth_year >= 1980

(4,598 real changes made)


In [179]:
egen birth_cohort_id = group(birth_cohort), label

In [180]:
tab birth_cohort_id year


group(birt |                          year
 h_cohort) |      2006       2008       2010       2012       2014 |     Total
-----------+-------------------------------------------------------+----------
 1935-1940 |       461        427        345        334        292 |     2,111 
 1940-1945 |       570        502        419        417        379 |     2,647 
 1945-1950 |       658        639        548        532        493 |     3,343 
 1950-1955 |     1,024      1,009        847        807        770 |     5,206 
 1955-1960 |     1,398      1,334      1,138      1,110      1,064 |     7,094 
 1960-1965 |     1,430      1,418      1,304      1,282      1,340 |     8,106 
 1965-1970 |     1,303      1,355      1,312      1,291      1,314 |     7,855 
 1970-1975 |       983      1,028      1,196      1,264      1,241 |     6,992 
 1975-1980 |       470        606        963      1,001      1,030 |     5,121 
     <1935 |       752        614        462        424        367 |     2,898

 `tab` cho thấy phần lớn cohort có quy mô mẫu ổn định qua các năm

## Phương án 2: Cohort theo năm sinh × giới tính chủ hộ

Cohort chi tiết hơn nhưng vẫn giữ quy mô mẫu khá lớn. Riêng một số nhóm năm sinh sau 1980 với chủ hộ là nữ (sexhead = 2) có khá ít quan sát ở các kỳ đầu và cuối của dữ liệu.

In [181]:
egen birth_sex_cohort_id = group(birth_cohort sexhead), label

In [182]:
tab birth_sex_cohort_id year


group(birth |
    _cohort |                          year
   sexhead) |      2006       2008       2010       2012       2014 |     Total
------------+-------------------------------------------------------+----------
1935-1940 1 |       301        271        195        181        151 |     1,225 
1935-1940 2 |       160        156        150        153        141 |       886 
1940-1945 1 |       358        310        227        219        205 |     1,517 
1940-1945 2 |       212        192        192        198        174 |     1,130 
1945-1950 1 |       444        425        367        346        317 |     2,178 
1945-1950 2 |       214        214        181        186        176 |     1,165 
1950-1955 1 |       723        718        626        570        525 |     3,657 
1950-1955 2 |       301        291        221        237        245 |     1,549 
1955-1960 1 |     1,031        971        816        824        777 |     5,168 
1955-1960 2 |       367        363        322       

## Phương án 3: Cohort theo tỉnh

Hai phương án trên có $n_c$ từ trung bình đến lớn nhưng số lượng cohort thấp (11 và 22), tạo thành một bộ dữ liệu bảng với $N$ và $T$ đều nhỏ. Phân theo tỉnh nâng $N$ lên 63 trong khi $n_c$ vẫn đủ lớn (dưới 5% cohort có dưới 102 mẫu).

In [183]:
egen tinh_cohort_id = group(tinh), label

In [184]:
* Phân phối quy mô cohort theo bách phân vị
preserve
    contract tinh_cohort_id year, freq(obs_count)
    summarize obs_count, detail
restore





                          Frequency
-------------------------------------------------------------
      Percentiles      Smallest
 1%           84             72
 5%          102             72
10%          102             84       Obs                 378
25%          114             84       Sum of wgt.         378

50%          138                      Mean           148.0714
                        Largest       Std. dev.      54.68417
75%          162            420
90%          192            420       Variance       2990.358
95%          234            453       Skewness       2.942725
99%          420            456       Kurtosis       14.55408



## Collapse về cấp cohort x năm và khai báo bảng

Tính trung bình của `thubq` và `tobacexp` theo cohort x năm, sau đó `xtset` để khai báo cấu trúc bảng.

In [185]:
gen cohort_id = tinh_cohort_id

In [186]:
collapse (mean) thubq tobacexp, by(cohort_id year)

In [187]:
xtset cohort_id year


Panel variable: cohort_id (strongly balanced)
 Time variable: year, 2006 to 2016, but with gaps
         Delta: 1 unit


---

**Lưu ý:** với dữ liệu VHLSS gốc (mỗi năm là một tệp riêng), cần lặp quy trình trên cho mỗi năm rồi `append` các tệp cohort lại trước khi `xtset`. Trong đó, cần bảo đảm có biến `year` thống nhất và mã cohort gán theo cùng quy tắc qua các năm.